In [1]:
import pandas as pd
from langchain_ollama import ChatOllama
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import BleuScore, RougeScore

/Users/akashdhande/miniconda3/envs/temp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------
# 1. Build the Evaluation Dataset
# ------------------------------

# Load the dataset from a CSV file
df = pd.read_csv(
    "../results/generated/Question_Answers_300_0.8_temp_singleline_nomic-embed-text_latest_deepseek-r1_7b.csv"
)

# Build the evaluation dataset using the stored columns.
dataset = []

for idx, row in df.iterrows():
    query = row["Question"]
    print(f"Processing row {idx+1} with query: {query}")

    # Reconstruct the retrieved chunks list.
    # We assume that the stored "Retrieved Chunks" column is a string of chunks separated by "\n\n".
    if isinstance(row["Retrieved Chunks"], str):
        retrieved_contexts = [
            chunk.strip()
            for chunk in row["Retrieved Chunks"].split("\n\n")
            if chunk.strip()
        ]
    else:
        retrieved_contexts = row["Retrieved Chunks"]

    # Build the sample dictionary.
    sample = {
        "user_input": row["Question"],
        "retrieved_contexts": retrieved_contexts,
        "response": row["Generated response"],
        "reference": row["Answer"],
    }
    dataset.append(sample)

# Create an EvaluationDataset from the list of samples.
evaluation_dataset = EvaluationDataset.from_list(dataset)

Processing row 1 with query: What are the key features of in-line centrifugal fans used in construction?
Processing row 2 with query: Why is efficient waste management important in construction projects?
Processing row 3 with query: What is the minimum compressive strength required for extruded polystyrene foam-plastic board used in masonry cavity insulation?
Processing row 4 with query: What are the minimum design wind pressures that exterior doors and frames must withstand?
Processing row 5 with query: What is the required annular clear space for sleeves used with sleeve-seal systems in concrete slabs and walls?
Processing row 6 with query: Why is it necessary to proceed with selective demolition systematically from higher to lower levels?
Processing row 7 with query: What are the qualification requirements for installers and testing agencies for steel decking?
Processing row 8 with query: Why is it important to maintain concrete temperature below 90Â°F during hot-weather placement?


In [3]:
# Check the dataset
print(f"Loaded {len(evaluation_dataset)} samples from the dataset.")
print(f"First sample: {evaluation_dataset[0]}")

Loaded 300 samples from the dataset.
First sample: user_input='What are the key features of in-line centrifugal fans used in construction?' retrieved_contexts=['Source (wendel-secifications_chunk_881.txt): , side wall, or ceiling mounting. b. direct - drive units : open type ec motor mounted in airstream, factory wired to disconnect switch located on outside of fan housing. motors permanently lubricated, with heavy duty ball bearings. controllable to 20 % of full speed ( 80 % turndown ). c. fan wheels : aluminum, airfoil blades welded to aluminum hub. d. accessories : 1. variable - speed controller : solid - state control to reduce speed from 100 to less than 50 %. 2. companion flanges : for inlet and outlet duct connections. 2. 4 motors a. refer to specification section 230513 – common motor requirements for hvac equipment. hvac fans 233400 - 2 mart office building renovations – phase 2 wendel project no. 610101 2. 5 source quality control a. sound - power level ratings : comply with 

In [4]:
# ------------------------------
# Run the Evaluation with the desired metrics.
# ------------------------------
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[BleuScore(), RougeScore(rouge_type="rougeL")],
)

Evaluating: 100%|██████████| 600/600 [00:00<00:00, 3808.65it/s]


In [5]:
print("Evaluation Results:")
print(result)

Evaluation Results:
{'bleu_score': 0.0620, 'rouge_score(mode=fmeasure)': 0.2097}


In [6]:
result.scores

[{'bleu_score': 0.0381671263989938,
  'rouge_score(mode=fmeasure)': 0.1764705882352941},
 {'bleu_score': 0.2271244283779713,
  'rouge_score(mode=fmeasure)': 0.3076923076923077},
 {'bleu_score': 0.5003635102626415,
  'rouge_score(mode=fmeasure)': 0.4444444444444444},
 {'bleu_score': 0.16077011415836423,
  'rouge_score(mode=fmeasure)': 0.36666666666666664},
 {'bleu_score': 0.44419850050604487,
  'rouge_score(mode=fmeasure)': 0.7777777777777777},
 {'bleu_score': 0.24601372576927547,
  'rouge_score(mode=fmeasure)': 0.3370786516853933},
 {'bleu_score': 0.026779624945488485,
  'rouge_score(mode=fmeasure)': 0.15384615384615383},
 {'bleu_score': 0.35125022525986127,
  'rouge_score(mode=fmeasure)': 0.3789473684210526},
 {'bleu_score': 0.36104927352363025,
  'rouge_score(mode=fmeasure)': 0.3529411764705882},
 {'bleu_score': 0.042852465180101385,
  'rouge_score(mode=fmeasure)': 0.372093023255814},
 {'bleu_score': 0.024101234231485975,
  'rouge_score(mode=fmeasure)': 0.45714285714285713},
 {'bleu_